In [ ]:
"""
# Process Monitor Database Logging Utility

This utility provides a production-ready function for recording process monitoring data in a PostgreSQL 
database. It handles database connections, data validation, and insertion into the `process_monitor_logs` table.

## Purpose and Benefits

This module enables comprehensive monitoring of LLM processing stages by:
- Recording execution times, token usage, and costs for each processing stage
- Tracking the decision path through multi-agent systems
- Providing data for performance analysis and optimization
- Enabling alerting when metrics exceed thresholds
- Supporting dashboard visualization of model performance

## Technical Implementation

The implementation follows these principles:
- Self-contained connection handling with configurable credentials
- Robust data validation before database operations
- Automatic calculation of derived fields (duration, costs, token counts)
- Full SQL transaction support with proper error handling
- Parameterized queries to prevent SQL injection

## Usage Guide

1. Configure the database connection parameters in the `DB_PARAMS` dictionary
2. Call `insert_process_monitor_entry()` with at minimum the required parameters:
   - `run_uuid`: Unique identifier for the entire processing run
   - `model_name`: Name of the LLM or processing model
   - `stage_name`: Name of the specific processing stage
   - `stage_start_time`: When the processing stage began

3. Optional fields provide deeper insights:
   - `stage_end_time`: When the stage completed
   - `llm_calls`: Detailed information about each LLM API call
   - `total_tokens`: Token usage metrics for analysis
   - `status`: Outcome of the stage
   - `decision_details`: Information about routing decisions
"""

import json
import uuid
import logging
import datetime
from typing import Dict, Any, List, Optional, Union
import psycopg2
from psycopg2.extras import register_uuid, Json

# Configure logging with timestamp, level and message format
logging.basicConfig(
    level=logging.INFO,
    format='%(asctime)s - %(levelname)s - %(message)s'
)
logger = logging.getLogger(__name__)

# Register UUID adapter for PostgreSQL to handle UUID objects properly
register_uuid()

# ------------------------------------------------------
# DATABASE CONFIGURATION 
# ------------------------------------------------------
# Database connection parameters - modify these to match your environment
DB_PARAMS = {
    "host": "localhost",
    "port": "5432",
    "dbname": "maven-finance",
    "user": "iris_dev",
    "password": "",  # Add password if required
}
# ------------------------------------------------------

def connect_to_db() -> Optional[psycopg2.extensions.connection]:
    """
    Establishes a connection to the PostgreSQL database using configured parameters.
    
    This function attempts to connect to the database using the parameters defined in the
    DB_PARAMS dictionary. It includes robust error handling and logging to facilitate
    troubleshooting of connection issues.
    
    Returns:
        psycopg2.extensions.connection: Active database connection object if successful
        None: If connection attempt fails for any reason
        
    Raises:
        No exceptions are raised; all exceptions are caught and logged
    """
    try:
        # Log connection attempt with sanitized parameters (no password)
        logger.info(f"Connecting to database with parameters: host={DB_PARAMS['host']}, "
                    f"port={DB_PARAMS['port']}, dbname={DB_PARAMS['dbname']}, user={DB_PARAMS['user']}")
        
        # Establish connection with all parameters
        conn = psycopg2.connect(**DB_PARAMS)
        
        # Set autocommit to False to enable transaction control
        conn.autocommit = False
        
        logger.info("Database connection successfully established")
        return conn
    except psycopg2.OperationalError as e:
        # Specific handling for connection failures like unreachable server or authentication issues
        logger.error(f"Database connection failed - operational error: {e}")
        return None
    except Exception as e:
        # Catch-all for any other unexpected errors
        logger.error(f"Unexpected error establishing database connection: {e}")
        return None

def insert_process_monitor_entry(
    # Required fields
    run_uuid: Union[uuid.UUID, str],
    model_name: str,
    stage_name: str,
    stage_start_time: datetime.datetime,
    # Optional fields with sensible defaults
    stage_end_time: Optional[datetime.datetime] = None,
    duration_ms: Optional[int] = None,
    llm_calls: Optional[List[Dict[str, Any]]] = None,
    total_tokens: Optional[int] = None,
    total_cost: Optional[float] = None,
    status: Optional[str] = None,
    decision_details: Optional[str] = None,
    error_message: Optional[str] = None,
    user_id: Optional[str] = None,
    environment: Optional[str] = None,
    custom_metadata: Optional[Dict[str, Any]] = None,
    notes: Optional[str] = None
) -> bool:
    """
    Inserts a process monitoring entry into the process_monitor_logs table.
    
    This function handles the complete lifecycle of a process monitoring entry insert:
    1. Validates required fields and data formats
    2. Performs type conversions where necessary (e.g., string UUID to UUID object)
    3. Calculates derived fields when not provided (duration, token counts, costs)
    4. Manages database connections, transactions, and proper cleanup
    5. Provides detailed logging for successful operations and failures
    
    The function is designed to be robust against invalid inputs and database connectivity
    issues, always returning a boolean status rather than raising exceptions.
    
    Args:
        run_uuid (Union[uuid.UUID, str]): 
            A UUID object or string identifying the entire processing run. 
            All stages within the same run should share the same UUID.
        
        model_name (str): 
            Identifier for the model being used (e.g., 'gpt-4', 'iris', 'claude').
            Limited to 100 characters by database schema.
        
        stage_name (str): 
            Name of the specific process stage (e.g., 'Router_Processing', 'Database_Query').
            Limited to 100 characters by database schema.
        
        stage_start_time (datetime.datetime): 
            The timestamp when this processing stage began.
            Should include timezone information (preferably UTC).
        
        stage_end_time (Optional[datetime.datetime]): 
            The timestamp when this processing stage ended.
            If provided with stage_start_time but no duration_ms, the duration is calculated.
        
        duration_ms (Optional[int]): 
            Duration of the stage in milliseconds.
            Calculated from start and end times if not provided but both times exist.
        
        llm_calls (Optional[List[Dict[str, Any]]]): 
            List of dictionaries with details of LLM API calls made during this stage.
            Expected format: [
                {
                    "model": str,           # The specific model used for this call
                    "input_tokens": int,    # Number of input/prompt tokens
                    "output_tokens": int,   # Number of output/completion tokens
                    "cost": float,          # Cost in USD for this specific call
                    "response_time_ms": int # Time taken for the LLM to respond in ms
                },
                ...
            ]
        
        total_tokens (Optional[int]): 
            Total token usage for this stage.
            If not provided but llm_calls is, this is calculated from the calls data.
        
        total_cost (Optional[float]): 
            Total cost in USD for this stage.
            If not provided but llm_calls is, this is calculated from the calls data.
        
        status (Optional[str]): 
            Status code or description of the stage outcome (e.g., 'Success', 'Failure', 'Timeout').
            Limited to 255 characters by database schema.
        
        decision_details (Optional[str]): 
            Text field for specific outputs or decisions made during this stage.
            For example, a router might record which agent it selected.
        
        error_message (Optional[str]): 
            Detailed error message if the stage failed.
            Only populated when status indicates a failure.
        
        user_id (Optional[str]): 
            Identifier for the user who initiated the request.
            Limited to 255 characters by database schema.
        
        environment (Optional[str]): 
            Deployment environment identifier (e.g., 'production', 'staging', 'development').
            Limited to 50 characters by database schema.
        
        custom_metadata (Optional[Dict[str, Any]]): 
            Flexible dictionary for any additional structured metadata.
            Stored as JSONB in the database for efficient querying.
        
        notes (Optional[str]): 
            Free-form text field for additional context or annotations.
        
    Returns:
        bool: True if the insert operation was successful, False otherwise
    """
    # Validations for required fields
    if not run_uuid or not model_name or not stage_name or not stage_start_time:
        logger.error("Missing required fields: run_uuid, model_name, stage_name, and stage_start_time are required")
        return False
    
    # Ensure stage_name and model_name don't exceed maximum lengths (100 chars)
    if len(model_name) > 100:
        logger.error(f"model_name exceeds maximum length of 100 characters: {len(model_name)}")
        return False
    
    if len(stage_name) > 100:
        logger.error(f"stage_name exceeds maximum length of 100 characters: {len(stage_name)}")
        return False
    
    # Convert string UUID to UUID object if needed
    if isinstance(run_uuid, str):
        try:
            run_uuid = uuid.UUID(run_uuid)
        except ValueError as e:
            logger.error(f"Invalid UUID format: {e}")
            return False
    
    # Ensure timestamps have timezone information
    if stage_start_time.tzinfo is None:
        logger.warning("stage_start_time has no timezone information; assuming UTC")
        stage_start_time = stage_start_time.replace(tzinfo=datetime.timezone.utc)
    
    if stage_end_time is not None and stage_end_time.tzinfo is None:
        logger.warning("stage_end_time has no timezone information; assuming UTC")
        stage_end_time = stage_end_time.replace(tzinfo=datetime.timezone.utc)
    
    # Calculate duration if not provided but have start and end times
    if duration_ms is None and stage_end_time is not None:
        try:
            # Calculate duration in milliseconds from timestamps
            duration_ms = int((stage_end_time - stage_start_time).total_seconds() * 1000)
            logger.debug(f"Calculated duration_ms from timestamps: {duration_ms}")
        except Exception as e:
            logger.warning(f"Could not calculate duration from timestamps: {e}. Will insert NULL for duration_ms.")
    
    # Process llm_calls if provided to calculate tokens and costs
    if llm_calls is not None:
        # Validate llm_calls structure
        if not isinstance(llm_calls, list):
            logger.error("llm_calls must be a list of dictionaries")
            return False
        
        # Calculate total token usage if not provided
        if total_tokens is None:
            try:
                total_tokens = sum(
                    (call.get('input_tokens', 0) + call.get('output_tokens', 0))
                    for call in llm_calls if isinstance(call, dict)
                )
                logger.debug(f"Calculated total_tokens from llm_calls: {total_tokens}")
            except Exception as e:
                logger.warning(f"Error calculating total tokens from llm_calls: {e}")
        
        # Calculate total cost if not provided
        if total_cost is None:
            try:
                total_cost = sum(
                    call.get('cost', 0)
                    for call in llm_calls if isinstance(call, dict)
                )
                logger.debug(f"Calculated total_cost from llm_calls: {total_cost}")
            except Exception as e:
                logger.warning(f"Error calculating total cost from llm_calls: {e}")
    
    # Connect to the database
    conn = connect_to_db()
    if not conn:
        logger.error("Failed to establish database connection. Cannot insert monitor entry.")
        return False
    
    # Insert the entry within a transaction
    try:
        with conn.cursor() as cur:
            query = """
            INSERT INTO process_monitor_logs (
                run_uuid, model_name, stage_name, stage_start_time, stage_end_time,
                duration_ms, llm_calls, total_tokens, total_cost, status,
                decision_details, error_message, user_id, environment, custom_metadata, notes
            ) VALUES (
                %s, %s, %s, %s, %s, %s, %s, %s, %s, %s, %s, %s, %s, %s, %s, %s
            ) RETURNING log_id
            """
            
            # Prepare JSONB fields with psycopg2's Json adapter
            llm_calls_jsonb = Json(llm_calls) if llm_calls is not None else None
            custom_metadata_jsonb = Json(custom_metadata) if custom_metadata is not None else None
            
            # Execute the query with all parameters
            cur.execute(query, (
                run_uuid, model_name, stage_name, stage_start_time, stage_end_time,
                duration_ms, llm_calls_jsonb, total_tokens, total_cost, status,
                decision_details, error_message, user_id, environment, custom_metadata_jsonb, notes
            ))
            
            # Get the inserted log_id from the RETURNING clause
            result = cur.fetchone()
            log_id = result[0] if result else None
            
            # Commit the transaction
            conn.commit()
            
            logger.info(f"Successfully inserted process monitor entry with log_id: {log_id}")
            return True
            
    except psycopg2.DataError as e:
        # Data type or format issues
        conn.rollback()
        logger.error(f"Data format error inserting process monitor entry: {e}")
        return False
    except psycopg2.IntegrityError as e:
        # Constraint violations
        conn.rollback()
        logger.error(f"Integrity constraint violation inserting process monitor entry: {e}")
        return False
    except Exception as e:
        # Catch all other errors
        conn.rollback()
        logger.error(f"Unexpected error inserting process monitor entry: {e}")
        return False
    finally:
        # Always close the connection
        conn.close()

def verify_process_monitor_table_exists() -> bool:
    """
    Verifies that the process_monitor_logs table exists in the connected database.
    
    This function checks the database information schema to confirm the existence of
    the process_monitor_logs table in the public schema. This verification helps
    diagnose setup issues before attempting to insert records.
    
    The function establishes its own database connection and properly closes it
    regardless of the outcome of the verification.
    
    Returns:
        bool: True if the table exists, False if it doesn't exist or can't be verified
              due to connection issues or insufficient privileges
    """
    conn = connect_to_db()
    if not conn:
        logger.error("Cannot verify table existence - database connection failed")
        return False
    
    try:
        with conn.cursor() as cur:
            # SQL query to check if the table exists in the information schema
            cur.execute("""
                SELECT EXISTS (
                    SELECT FROM information_schema.tables 
                    WHERE table_schema = 'public'
                    AND table_name = 'process_monitor_logs'
                );
            """)
            # Extract the boolean result
            exists = cur.fetchone()[0]
            return exists
    except psycopg2.Error as e:
        logger.error(f"Database error while checking if table exists: {e}")
        return False
    except Exception as e:
        logger.error(f"Unexpected error checking if table exists: {e}")
        return False
    finally:
        conn.close()

# Automatically verify table existence when the module is loaded
table_exists = verify_process_monitor_table_exists()
if table_exists:
    logger.info("✅ process_monitor_logs table verified - ready for data insertion")
else:
    logger.warning("⚠️ process_monitor_logs table not found in the database")
    logger.info("Execute the table creation SQL from postgres_schema.sql before attempting to insert records")

In [ ]:
"""
# Process Monitor Usage Examples
  
This cell demonstrates how to use the process monitoring utility to track performance,
execution times, and resource utilization across different stages of model processing.

## Example Overview

The examples below show two common usage patterns:

1. **Complete Monitoring Record** - Records comprehensive metrics including:
   - Full timing information (start/end timestamps, calculated duration)
   - Detailed LLM API call data (tokens, costs, response times)
   - Status information and decision details
   - Custom metadata for additional context
   
2. **Minimal Monitoring Record** - Shows the simplest possible usage with only
   the required fields, useful for lightweight tracking or when full details
   aren't available yet.

These patterns can be applied at various stages of model execution:
- Initial request processing
- Router/dispatcher decision points
- Individual agent processing steps
- Database query operations
- Final response generation

The collected data enables performance analysis, anomaly detection, cost tracking,
and can feed into monitoring dashboards and alerting systems.
"""

# ---------------------------------------------------------------------------------
# Example Data - Fixed values for demonstration purposes
# ---------------------------------------------------------------------------------

# Fixed example values for demonstration and testing purposes
EXAMPLE_RUN_UUID = "123e4567-e89b-12d3-a456-426614174000"  # Fixed example UUID
EXAMPLE_MODEL = "gpt-4"                                     # Model identifier 
EXAMPLE_STAGE = "router_processing"                         # Process stage name

# Example of LLM API call details
EXAMPLE_LLM_CALLS = [
    {
        "model": "gpt-4",
        "input_tokens": 550,  # Prompt tokens
        "output_tokens": 320, # Completion tokens
        "cost": 0.0265,       # Cost in USD
        "response_time_ms": 2500  # Response time in milliseconds
    }
]

# Example of custom metadata
EXAMPLE_METADATA = {
    "request_id": "req_0123456789abcdef",
    "priority": "high",
    "client_type": "internal"
}

# Create timestamps for the example
current_time = datetime.datetime.now(datetime.timezone.utc)
start_time = current_time - datetime.timedelta(seconds=10)  # 10 seconds ago
end_time = current_time  # Now

# ---------------------------------------------------------------------------------
# EXAMPLE 1: Complete Process Monitor Entry (All Fields)
# ---------------------------------------------------------------------------------
print("📝 INSERTING COMPLETE PROCESS MONITOR ENTRY")
print(f"Run UUID:     {EXAMPLE_RUN_UUID}")
print(f"Model:        {EXAMPLE_MODEL}")
print(f"Stage:        {EXAMPLE_STAGE}")
print(f"Start time:   {start_time}")
print(f"End time:     {end_time}")
print(f"Duration:     {(end_time - start_time).total_seconds() * 1000:.0f} ms")
print(f"LLM calls:    {len(EXAMPLE_LLM_CALLS)} call(s)")
print(f"Total tokens: {EXAMPLE_LLM_CALLS[0]['input_tokens'] + EXAMPLE_LLM_CALLS[0]['output_tokens']} tokens")
print(f"Total cost:   ${EXAMPLE_LLM_CALLS[0]['cost']}")
print("-" * 50)

# Insert a fully populated entry
complete_entry = insert_process_monitor_entry(
    # Required fields
    run_uuid=EXAMPLE_RUN_UUID,
    model_name=EXAMPLE_MODEL,
    stage_name=EXAMPLE_STAGE,
    stage_start_time=start_time,
    
    # Optional fields - all populated for the complete example
    stage_end_time=end_time,
    duration_ms=int((end_time - start_time).total_seconds() * 1000),
    llm_calls=EXAMPLE_LLM_CALLS,
    total_tokens=EXAMPLE_LLM_CALLS[0]['input_tokens'] + EXAMPLE_LLM_CALLS[0]['output_tokens'],
    total_cost=EXAMPLE_LLM_CALLS[0]['cost'],
    status="Success",
    decision_details="Router selected database_query agent",
    error_message=None,
    user_id="example_user",
    environment="development",
    custom_metadata=EXAMPLE_METADATA,
    notes="Example entry with all fields populated"
)

if complete_entry:
    print("✅ Complete entry insertion successful!")
else:
    print("❌ Complete entry insertion failed!")

print("\n")

# ---------------------------------------------------------------------------------
# EXAMPLE 2: Minimal Process Monitor Entry (Required Fields Only)
# ---------------------------------------------------------------------------------
print("📝 INSERTING MINIMAL PROCESS MONITOR ENTRY")
print(f"Run UUID:     {EXAMPLE_RUN_UUID}")
print(f"Model:        {EXAMPLE_MODEL}")
print(f"Stage:        minimal_example")
print(f"Start time:   {current_time}")
print("-" * 50)

# Insert an entry with only the minimum required fields
minimal_entry = insert_process_monitor_entry(
    run_uuid=EXAMPLE_RUN_UUID,
    model_name=EXAMPLE_MODEL,
    stage_name="minimal_example",
    stage_start_time=current_time
)

if minimal_entry:
    print("✅ Minimal entry insertion successful!")
else:
    print("❌ Minimal entry insertion failed!")